# Spanish: Whisper full fine-tuning and LoRA

Runtime → Change runtime type → GPU. Run cells in order. Free GPU availability varies. Nothing uploads models or buys compute.

These cells call the package. Compare on the same untouched test clips; do not assume training improves WER.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/asr-finetuning-lab")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/jorgeutd/asr-finetuning-lab.git", str(repo)],
        check=True,
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}[train]"], check=True)
os.chdir(repo)
print(
    "Repository revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
)

In [ ]:
from datetime import UTC, datetime

import torch

from asr_lab.manifest import write_json

LANGUAGE = "es"
METHODS = ["lora", "full"]
STEPS = 300
PERSIST_TO_DRIVE = False

assert torch.cuda.is_available(), "Select a GPU runtime, then rerun. No paid runtime is required."
print(torch.cuda.get_device_name(0))
run_id = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
root = repo / "runs" / f"colab-{LANGUAGE}-{run_id}"
if PERSIST_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/asr-lab") / f"{LANGUAGE}-{run_id}"
root.mkdir(parents=True, exist_ok=False)
write_json(
    root / "execution.json",
    {
        "repository_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True
        ).strip(),
        "language": LANGUAGE,
        "methods": METHODS,
        "steps": STEPS,
    },
)

## Prepare data

FLEURS / CC BY 4.0. Keep the source splits. Speaker IDs are unavailable in the published fields. The selected prefix is a demonstration subset, not a representative benchmark.

In [ ]:
from asr_lab.data import fetch_fleurs
from asr_lab.manifest import validate_manifest

data = repo / "data" / "local" / LANGUAGE
manifest = data / "manifest.jsonl"
if not manifest.exists():
    fetch_fleurs(data, LANGUAGE, {"train": 256, "validation": 32, "test": 64})
validate_manifest(manifest)

## Baseline

Warm-up is excluded from timing. WER/CER use the same normalizer for every model.

In [ ]:
from asr_lab.evaluation import evaluate
from asr_lab.modeling import resolve_revision

MODEL = "openai/whisper-tiny"
REVISION = resolve_revision(MODEL, "main")
baseline = root / "baseline"
evaluate(MODEL, manifest, baseline, LANGUAGE, revision=REVISION)

## Train and evaluate

Validation WER selects the checkpoint. Test data is used only after training. LoRA learns small attention updates; full fine-tuning updates every weight.

For interrupted runs, keep the original recipe and use `train(recipe, manifest, run, resume=str(run / "checkpoint-100"))`. Drive persistence must be enabled before the session ends.

In [ ]:
import gc
from dataclasses import replace

from asr_lab.config import load_recipe
from asr_lab.reporting import compare
from asr_lab.training import train

comparisons = []
for method in METHODS:
    recipe = replace(
        load_recipe(repo / "recipes" / f"{LANGUAGE}-{method}.yaml"),
        revision=REVISION,
        max_steps=STEPS,
    )
    run = root / method
    training_result = train(recipe, manifest, run)
    print(training_result)
    gc.collect()
    torch.cuda.empty_cache()
    model = MODEL if method == "lora" else str(run / "model")
    adapter = str(run / "model") if method == "lora" else None
    candidate = root / f"{method}-eval"
    evaluate(model, manifest, candidate, LANGUAGE, revision=REVISION, adapter=adapter)
    comparisons.append(compare(baseline, candidate, root / f"{method}-comparison"))
    gc.collect()
    torch.cuda.empty_cache()
comparisons

## Inspect and save

Read errors as well as aggregate scores. A small test set and one training seed do not establish production quality. Download evidence before disconnecting. Review transcripts before publishing results.

In [ ]:
import json
import shutil

from google.colab import files

predictions = [
    json.loads(line)
    for line in (root / f"{METHODS[0]}-eval" / "predictions.jsonl").read_text().splitlines()
]
for row in predictions[:3]:
    print("Reference:", row["reference"])
    print("Prediction:", row["prediction"], "\n")

evidence = root / "evidence"
evidence.mkdir(exist_ok=True)
for path in root.rglob("*"):
    if (
        path.is_file()
        and "model" not in path.relative_to(root).parts
        and "evidence" not in path.parts
        and path.suffix in {".json", ".jsonl", ".md"}
        and not any(p.startswith("checkpoint-") for p in path.parts)
    ):
        target = evidence / path.relative_to(root)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
dataset_evidence = evidence / "dataset"
dataset_evidence.mkdir(exist_ok=True)
for name in ["manifest.jsonl", "provenance.json"]:
    shutil.copy2(data / name, dataset_evidence / name)
archive = shutil.make_archive(str(root / "evidence"), "zip", evidence)

files.download(archive)